# Step 9: Round 2 recording check

Step 8-এর `round2/wav_16k/` ফাইলগুলো ঠিক বাক্যের সঙ্গে মিলছে কি না দেখে, পরিষ্কার একটা তালিকা বানায়।

1. প্রতিটা রেকর্ডিং **fine-tuned Whisper-large-v3-turbo** (Round 1-এর সেরা মডেল) দিয়ে লেখা হয়।
2. প্রতিটা বক্তার জন্য আলাদাভাবে ফাইল নম্বর ↔ corpus-এর বাক্য নম্বরের **offset** বের করা হয় (−3 থেকে +3 পরীক্ষা করে)।
3. প্রতিটা রেকর্ডিংকে একটা রায় দেওয়া হয়: `ok`, `uncertain`, `moved` (অন্য বাক্যে সরানো), `bad` (বাদ)।
4. একই বাক্য দুবার রেকর্ড হলে কম ভুলের take-টা রাখা হয়।

**আউটপুট** (`Ultimate Thisis Data/round2/recheck/`): `transcripts.csv`, `round2_clean.csv` (training-এর জন্য তৈরি তালিকা), `report_round2.csv`।
কোনো অডিও ফাইল মোছা, সরানো বা বদলানো হয় না। GPU লাগবে (L4 বা A100)। মাঝপথে থামলে আবার চালালে আগের লেখা transcript থেকে শুরু করবে।

In [ ]:
# Cell 1: Drive mount + প্যাকেজ
from google.colab import drive
drive.mount('/content/drive')
!pip -q install jiwer rapidfuzz
print('ok')

In [ ]:
# Cell 2: সেটিং + ডেটা লোড
import os, re, unicodedata, numpy as np, pandas as pd
ROOT   = '/content/drive/MyDrive/Ultimate Thisis Data'
R2     = f'{ROOT}/round2'
OUT    = f'{R2}/recheck'
CORPUS = f'{ROOT}/corpus.csv'
ASR_CANDIDATES = ['/content/drive/MyDrive/sylheti_asr/whisper-large-v3-turbo-4spk/final',
                  '/content/drive/MyDrive/sylheti_asr/whisper-large-v3-turbo-4spk/checkpoint-2250']
os.makedirs(OUT, exist_ok=True)
ASR = next((p for p in ASR_CANDIDATES if os.path.exists(p)), None)
assert ASR, 'fine-tuned turbo মডেল পাওয়া যায়নি, ASR_CANDIDATES-এ ঠিক path দিন'
print('ASR model:', ASR)

meta = pd.read_csv(f'{R2}/metadata_round2.csv')
meta = meta[meta['flag'].fillna('') == ''].copy()          # <1s আর >30s বাদ
meta['abs_path'] = R2 + '/' + meta['path']
corpus = pd.read_csv(CORPUS)
print('round2 clips:', len(meta), '| corpus sentences:', len(corpus))
print(corpus[['no', 'excel_row', 'split']].head(3))

# Round 1-এর মতো একই clean(): NFC, zero-width বাদ, ZWJ (্য-এর আগেরটা ছাড়া) বাদ, যতিচিহ্ন বাদ
ZW = dict.fromkeys(map(ord, '\u200b\u200c\ufeff\u00ad\u2060'), None)
PUNCT = re.compile(r'["\'“”‘’(),.।?!:;\-/*…\[\]|]')
def clean(t):
    t = unicodedata.normalize('NFC', str(t)).translate(ZW)
    t = re.sub('\u200d(?!্য)', '', t)
    t = PUNCT.sub(' ', t)
    return re.sub(r'\s+', ' ', t).strip()
corpus['ref'] = corpus['sylheti'].map(clean)
ref_of = dict(zip(corpus['no'], corpus['ref']))

In [ ]:
# Cell 3: সব রেকর্ডিং লেখা (transcribe) — আগে লেখা থাকলে বাদ দেবে
import torch, librosa
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from tqdm.auto import tqdm
TR = f'{OUT}/transcripts.csv'
done = pd.read_csv(TR) if os.path.exists(TR) else pd.DataFrame(columns=['path', 'hyp'])
todo = meta[~meta['path'].isin(set(done['path']))].sort_values('duration_s')
print('বাকি:', len(todo), 'আগে হয়ে গেছে:', len(done))

device = 'cuda'
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
proc = WhisperProcessor.from_pretrained(ASR)
model = WhisperForConditionalGeneration.from_pretrained(ASR, torch_dtype=dtype).to(device).eval()
BATCH = 16
buf = []
for i in tqdm(range(0, len(todo), BATCH)):
    chunk = todo.iloc[i:i + BATCH]
    wavs = [librosa.load(p, sr=16000)[0] for p in chunk['abs_path']]
    feats = proc.feature_extractor(wavs, sampling_rate=16000, return_tensors='pt').input_features.to(device, dtype)
    with torch.no_grad():
        ids = model.generate(feats, language='bengali', task='transcribe', max_new_tokens=440)
    hyps = proc.batch_decode(ids, skip_special_tokens=True)
    buf += list(zip(chunk['path'], hyps))
    if len(buf) >= 400 or i + BATCH >= len(todo):          # মাঝে মাঝে Drive-এ সেভ
        done = pd.concat([done, pd.DataFrame(buf, columns=['path', 'hyp'])], ignore_index=True)
        done.to_csv(TR, index=False, encoding='utf-8-sig'); buf = []
print('মোট transcript:', len(done))

In [ ]:
# Cell 4: offset বের করা + প্রতিটা রেকর্ডিংয়ের রায়
import jiwer
from rapidfuzz import process, fuzz
tr = pd.read_csv(f'{OUT}/transcripts.csv')
df = meta.merge(tr, on='path', how='inner')
df['hyp_c'] = df['hyp'].fillna('').map(clean)

def cer(ref, hyp):
    if not ref: return 1.0
    return min(jiwer.cer(ref, hyp if hyp else '-'), 2.0)

# ---- প্রতি বক্তার offset (ফাইল নম্বর + offset = corpus 'no') ----
OFFSETS = range(-3, 4)
offset_of = {}
for spk, g in df.groupby('speaker'):
    sample = g.sample(min(150, len(g)), random_state=13)
    med = {o: np.median([cer(ref_of.get(n + o, ''), h) for n, h in zip(sample['file_num'], sample['hyp_c'])]) for o in OFFSETS}
    offset_of[spk] = min(med, key=med.get)
    print(spk, 'offset =', offset_of[spk], '| median CER:', {o: round(v, 2) for o, v in med.items()})
df['no_filed'] = df['file_num'] + df['speaker'].map(offset_of)
df['cer_filed'] = [cer(ref_of.get(n, ''), h) for n, h in zip(df['no_filed'], df['hyp_c'])]

# ---- পুরো corpus-এ সবচেয়ে কাছের বাক্য (ভুল জায়গায় রাখা রেকর্ডিং ধরার জন্য) ----
refs = corpus['ref'].tolist(); nos = corpus['no'].tolist()
best = [process.extractOne(h, refs, scorer=fuzz.ratio) if h else (None, 0, None) for h in tqdm(df['hyp_c'])]
df['no_best'] = [nos[b[2]] if b[2] is not None else None for b in best]
df['score_best'] = [b[1] for b in best]
df['score_filed'] = [fuzz.ratio(h, ref_of.get(n, '')) for n, h in zip(df['no_filed'], df['hyp_c'])]

# ---- রায় (thresholds: আমার বেছে নেওয়া, দরকার হলে বদলান) ----
OK_CER, UNC_CER = 0.40, 0.80
MOVE_MIN, MOVE_MARGIN = 70, 15
def verdict(r):
    if r.cer_filed <= OK_CER: return 'ok'
    if (r.no_best is not None and r.no_best != r.no_filed
            and r.score_best >= MOVE_MIN and r.score_best - r.score_filed >= MOVE_MARGIN): return 'moved'
    if r.cer_filed <= UNC_CER: return 'uncertain'
    return 'bad'
df['verdict'] = df.apply(verdict, axis=1)
df['no_final'] = np.where(df['verdict'] == 'moved', df['no_best'], df['no_filed'])
df.loc[df['verdict'] == 'moved', 'cer_final'] = [cer(ref_of.get(int(n), ''), h) for n, h in
                                                 zip(df.loc[df['verdict'] == 'moved', 'no_final'], df.loc[df['verdict'] == 'moved', 'hyp_c'])]
df['cer_final'] = df['cer_final'].fillna(df['cer_filed'])

# ---- একই বক্তা + একই বাক্যে একাধিক রেকর্ডিং হলে কম CER-এরটা রাখা ----
keep = df[df['verdict'] != 'bad'].sort_values('cer_final')
keep = keep.drop_duplicates(subset=['speaker', 'no_final'], keep='first')
df['kept'] = df.index.isin(keep.index)
print(df.groupby('speaker')['verdict'].value_counts().unstack(fill_value=0))

In [ ]:
# Cell 5: পরিষ্কার তালিকা + রিপোর্ট সেভ
c = corpus.set_index('no')
clean_df = df[df['kept']].copy()
clean_df['no'] = clean_df['no_final'].astype(int)
clean_df = clean_df.join(c[['excel_row', 'split', 'sylheti', 'bangla', 'english']], on='no')
cols = ['speaker', 'path', 'no', 'excel_row', 'split', 'sylheti', 'bangla', 'english',
        'duration_s', 'verdict', 'cer_final', 'file_num', 'take']
clean_df[cols].to_csv(f'{OUT}/round2_clean.csv', index=False, encoding='utf-8-sig')
df.drop(columns=['abs_path']).to_csv(f'{OUT}/recheck_all_round2.csv', index=False, encoding='utf-8-sig')

rep = df.groupby('speaker').agg(clips=('path', 'count'),
                                offset=('speaker', lambda s: offset_of[s.iloc[0]]),
                                ok=('verdict', lambda v: int((v == 'ok').sum())),
                                uncertain=('verdict', lambda v: int((v == 'uncertain').sum())),
                                moved=('verdict', lambda v: int((v == 'moved').sum())),
                                bad=('verdict', lambda v: int((v == 'bad').sum())),
                                kept=('kept', 'sum'),
                                median_cer=('cer_final', lambda x: round(float(np.median(x)) * 100, 1)))
hours = clean_df.groupby('speaker')['duration_s'].sum().div(3600).round(2).rename('kept_hours')
rep = rep.join(hours).reset_index()
rep.loc[len(rep)] = ['TOTAL', rep.clips.sum(), '', rep.ok.sum(), rep.uncertain.sum(), rep.moved.sum(),
                     rep.bad.sum(), rep.kept.sum(), '', round(rep.kept_hours.sum(), 2)]
rep.to_csv(f'{OUT}/report_round2.csv', index=False, encoding='utf-8-sig')
print(rep.to_string(index=False))
print('\nsplit অনুযায়ী রাখা রেকর্ডিং:')
print(clean_df.groupby(['speaker', 'split']).size().unstack(fill_value=0))
print('\nসেভ হয়েছে:', f'{OUT}/round2_clean.csv')